# Streaming

> Streaming helpers for lossless event collation.

In [ ]:
#| default_exp streaming

## Imports

In [ ]:
#| export
import json
from dataclasses import dataclass, field, fields
from fastcore.utils import *
from fastcore.meta import delegates
from fastspec.errors import *
from fastllm.types import *

In [ ]:
#| hide
import yaml,json
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from cachy.core import enable_cachy, disable_cachy, doms
from IPython.core.display import Markdown
from importlib.resources import files

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
specs_path = files('fastllm') / 'specs'
ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
kimi_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"})
for op in kimi_cli.ops: op.base_url = 'https://api.fireworks.ai/inference/v1'

In [ ]:
ant_spec, oai_spec, gem_spec

(SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

### Delta

Canonical streaming event fragment

In [ ]:
#| export
@dataclass
class Delta:
    "Normalized streaming delta event."
    text: str = ""
    thinking: str = ""
    refusal: str = ""
    tool_calls: list[ToolCall] = field(default_factory=list)
    citations: list = field(default_factory=list)
    server_tool_result: dict = None
    finish_reason: str = None
    usage: Usage = None
    raw: dict = field(default_factory=dict)

In [ ]:
#| export
async def norm_and_yield(resp, norm_func):
    async for ev in resp:
        if (d := norm_func(ev)) is not None: yield d

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o):
        if not isinstance(o, Completion): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

`ToolCall` normalizes a tool call with `id`, `name`, and `arguments`, anything extra goes into `extra`

`tool_use` parts are built from tool calls, and store `ToolCall` fields in `data`, which will be converted into correct message type based on provider, and allow users to pass `Msg` directly to the chat history

### PartAccum

Collator for tool calls and other text parts

In [ ]:
#| export
@dataclass
class PartAccum:
    parts: dict[Part|ToolCall] = field(default_factory=dict)
    tool_calls: list[ToolCall] = field(default_factory=list)
    
    def append(self, typ, index, txt='', citations=None, **tc_kwargs):
        'Create and accumulate same type sequential parts'
        if index not in self.parts: 
            if typ==PartType.tool_use: self.parts[index] = ToolCall(**tc_kwargs)
            else:                      self.parts[index] = Part(type=typ, text=txt, data=dict(citations=citations or []))
        else:
            if typ==PartType.tool_use:
                new_args = tc_kwargs.get('arguments', '')
                cur_args = self.parts[index].arguments
                if isinstance(new_args, str) and isinstance(cur_args, str): self.parts[index].arguments += new_args
                elif isinstance(new_args, str) and isinstance(cur_args, dict): self.parts[index].arguments = new_args
                else: self.parts[index].arguments = new_args
            else: 
                self.parts[index].text += txt
                # anthropic citations have matching idx
                self.parts[index].data['citations'].extend(citations or [])
    
    def finalize(self):
        for idx,tc in self.parts.items():
            if isinstance(tc, ToolCall):
                if isinstance(tc.arguments, str): tc.arguments = json.loads(tc.arguments) if tc.arguments else {}
                self.tool_calls.append(tc)
                data = {**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server}
                self.parts[idx] = Part(type=PartType.tool_use, data=data)
        
        merged = []
        for p in self.parts.values():
            if merged and merged[-1].type == p.type and p.type in (PartType.text, PartType.thinking): merged[-1].text += p.text
            else: merged.append(p)        
        self.parts = merged

### mk_acollect_stream

A generic `mk_acollect_stream` function that yields thinking and text (if needed we can yield tool calls to), and collates parts while keeping the order. A custom `index_fn` is used to resolve the index that the current `Delta` event belongs to.

In [ ]:
#| export
def accum_completion(pa, raw, fin, usg, deltas, model=None, api_name=None, vendor_name=None, delta=None):
    "Build a Completion snapshot from in-progress PartAccum state"
    parts = [p for p in pa.parts.values() if isinstance(p, Part)]
    if delta and delta.text:
        parts = parts.copy()
        if parts and parts[-1].type==PartType.text:
            p = parts[-1]
            parts[-1] = Part(type=p.type, text=(p.text or '') + delta.text, data=p.data)
        else: parts.append(Part(type=PartType.text, text=delta.text))
    return Completion(raw.get('model', model), Msg(role="assistant", content=parts),
                      fin, usg, api_name=api_name, vendor_name=vendor_name, raw={'deltas':deltas})

In [ ]:
#| export
def completion_text(c):
    "Combined text from a Completion."
    return ''.join(p.text or '' for p in c.message.content if p.type==PartType.text)

Joins all `text`-type parts in the completion's message into a single string. Used by stop-sequence checking and anywhere plain output text is needed — `thinking` and tool-use parts are ignored.

Mixed parts — only `text` content is concatenated:

In [ ]:
c = Completion(None, Msg(role="assistant", content=[
    Part(type=PartType.thinking, text="reasoning..."),
    Part(type=PartType.text, text="Hello "),
    Part(type=PartType.text, text="world"),
]))
test_eq(completion_text(c), "Hello world")

In [ ]:
#| export
def stop_sequences(seqs):
    "Stop when any sequence appears in the accumulated completion text."
    seqs = L(seqs)
    def _stop(c):
        txt = completion_text(c)
        for s in seqs:
            if s in txt: return s
    return _stop

In [ ]:
#| export
def _trim_delta(d, cur, s):
    "Trim `d.text` so accumulated text in `cur` stops just before stop sequence `s`."
    txt,dt = completion_text(cur), d.text or ''
    i = txt.find(s)
    if i>=0: d.text = dt[:max(0, i-(len(txt)-len(dt)))]

This is a helper for stop-sequence handling. When a stop callable detects its sequence mid-stream, `_trim_delta` rewrites the current delta's `text` so the accumulated completion ends just before the matched sequence — keeping any prefix already emitted in earlier deltas, and emitting nothing for this delta if the sequence started in earlier text.

Stop sequence "STOP" begins mid-delta — only the prefix should remain:

In [ ]:
d = Delta(text=" world STOP after")
cur = Completion(None, Msg(role="assistant", content=[Part(type=PartType.text, text="Hello world STOP after")]))
_trim_delta(d, cur, "STOP")
test_eq(d.text, " world ")

Stop sequence began in earlier text — current delta trimmed to empty:

In [ ]:
d = Delta(text="ello world")
cur = Completion(None, Msg(role="assistant", content=[Part(type=PartType.text, text="Say hello world")]))
_trim_delta(d, cur, "hello")
test_eq(d.text, "")

In [ ]:
#| export
async def mk_acollect_stream(it, index_fn, model=None, api_name=None, vendor_name=None, stop_callables=None):
    "Collect a Delta stream, yielding incremental chunks and a final Completion."
    part_accum,deltas,stop_callables = PartAccum(), [], L(stop_callables)
    fin, usg, typ, last_typ, last_idx = [None]*5
    def _fidx(d, name, pt=None):
        nonlocal typ, last_idx
        typ = getattr(PartType, pt or name)
        idx,last_idx = index_fn(d, typ, last_typ, last_idx)
        return idx
    def _proc(d, name, pt=None, kw='txt', ret=None):
        if not ret and not (val := getattr(d, name)): return
        idx = _fidx(d, name, pt)
        part_accum.append(typ, idx, **(ret or {kw: val}))
        return ret or {name: val}
    async for d in it:
        stop = False
        if stop_callables:
            cur = accum_completion(part_accum, d.raw, fin, usg, deltas+[d], model, api_name=api_name, vendor_name=vendor_name, delta=d)
            for f in stop_callables:
                if res:=f(cur):
                    if isinstance(res, str): _trim_delta(d, cur, res)
                    stop = True
                    break
        if (r:=_proc(d, 'text')): yield r
        if (r:=_proc(d, 'thinking')): yield r
        if (r:=_proc(d, 'citations', pt='text', kw='citations')): yield r
        for tc in d.tool_calls:
            args = tc.arguments.get('_delta', tc.arguments)
            _proc(d, 'tool_use', ret=dict(id=tc.id, name=tc.name, arguments=args, server=tc.server, extra=tc.extra))
        if d.server_tool_result:
            idx = _fidx(d, 'server_tool_result')
            part_accum.parts[idx] = Part(type=typ, data=d.server_tool_result)
        if (r:=_proc(d, 'refusal')): yield r
        if d.finish_reason: fin = d.finish_reason
        if d.usage: usg = d.usage
        last_typ = typ
        deltas.append(d)
        if stop:
            fin = fin or FinishReason.stop
            await it.aclose()
            break
    part_accum.finalize()
    # need to recheck for tool calls post collation for streaming
    tcs = part_accum.tool_calls
    fin = FinishReason.tool_calls if fin==FinishReason.stop and any(~L(tcs).attrgot('server')) else fin
    # tool calls and non-anthropic citations are yielded at the end
    yield Completion(d.raw.get('model', model),
            message=Msg(role="assistant", content=part_accum.parts),
            finish_reason=fin, usage=usg, tool_calls=tcs, api_name=api_name, vendor_name=vendor_name,
            raw={'deltas':deltas})

`stop_callables` preview the partial `Completion` including the current text delta. Returning a string stops and trims from that string before yielding the delta; `stop_sequences` builds that callable for common string stops.


In [ ]:
#| export
async def fake_stream(*ss):
    for s in ss: yield Delta(text=s, raw={'model':'fake'})

In [ ]:
def _stop_hello(c): return c.message.content[0].text == 'hello'

strm = fake_stream('he','llo','!')
chunks = [o async for o in mk_acollect_stream(strm, lambda *args: (0,0), stop_callables=_stop_hello)]

test_eq(chunks[:-1], [{'text':'he'}, {'text':'llo'}])
test_eq(chunks[-1].message.content[0].text, 'hello')
test_eq(chunks[-1].finish_reason, FinishReason.stop)


strm = fake_stream('ab','cSTOP','!')
chunks = [o async for o in mk_acollect_stream(strm, lambda *args: (0,0), stop_callables=stop_sequences('STOP'))]

test_eq(chunks[:-1], [{'text':'ab'}, {'text':'c'}])
test_eq(chunks[-1].message.content[0].text, 'abc')
strm = fake_stream('ab','cSTOP','!')
chunks = [o async for o in mk_acollect_stream(strm, lambda *args: (0,0), stop_callables=stop_sequences('STOP'))]

test_eq(chunks[:-1], [{'text':'ab'}, {'text':'c'}])
test_eq(chunks[-1].message.content[0].text, 'abc')

In [ ]:
async def _fake_tool_stream(*ss):
    for s in ss: yield Delta(tool_calls=[ToolCall(id='1', name='simple_add', arguments={'_delta':s})], raw={'model':'fake'})
    yield Delta(finish_reason=FinishReason.stop, raw={'model':'fake'})

def _idx_fn(d,typ,lt,li): return (d.tool_calls[0].id, li) if d.tool_calls else (0, li)

strm = _fake_tool_stream('{"a":', '1,"b":2}')
chunks = [o async for o in mk_acollect_stream(strm, _idx_fn)]
test_eq(chunks[-1].tool_calls[0].arguments, {'a':1, 'b':2})
test_eq(chunks[-1].tool_calls[0].name, 'simple_add')
test_eq(chunks[-1].finish_reason, FinishReason.tool_calls)

In [ ]:
async def _fake_srv_stream(res):
    yield Delta(server_tool_result=res, raw={'model':'fake'})
    yield Delta(finish_reason=FinishReason.stop, raw={'model':'fake'})

strm = _fake_srv_stream({'output':'42'})
chunks = [o async for o in mk_acollect_stream(strm, lambda *a: (0,0))]
p = chunks[-1].message.content[0]
test_eq(p.type, PartType.server_tool_result)
test_eq(p.data, {'output':'42'})

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()